### Reopening the index
In a new Python session, you can reopen the index without re-computing embeddings:

In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Using sqlitesearch vector search in RAG

Let's use our persistent vector index in RAG.

In a new notebook, set up the model and open the index (same as the "Reopening the index" section above):

We'll use the RAGVector class we defined in the previous lesson. It overrides the search method to embed the query and use vector search.

Set up the OpenAI client and create the assistant:

In [2]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

Try it:

In [3]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join.\n\nYou don’t need to wait for a confirmation email or a separate approval. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

When you're done, close the connection:

In [4]:
vs_index.close()